### Exercise 1

A CSVReader class should encapsulate all functionality related to reading CSV files.

Instructions:

    1. Create a class CSVReader.

    2. Define an __init__ method that takes file_path.

    3. Add a read() method that loads the CSV file using pandas.

    4. Add a preview(n) method that prints the top n rows.

In [1]:
import pandas as pd

class CSVReader:
    def __init__(self, filepath: str):
        self.file_path = filepath

    def read(self): 
        df = pd.read_csv(self.file_path)
        return df

    def preview(self, df, n):
        return df.head(n)


read_csv = CSVReader(filepath="./sample_data.csv")
df = read_csv.read()
read_csv.preview(df=df, n=10)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


### Exercise 2: Apply Pattern for Missing Values Handling

Objective:

    • Learn the Strategy Pattern - one interface, many interchangeable behaviors.
    • Practice inheritance and polymorphism.

Concept:
We often need different strategies to handle missing data (drop, fill mean, fill mode, etc.).
The pattern allows flexible switching between methods without changing the main code.

Instructions:

    1. Create an abstract class MissingValueStrategy with a method handle(df).
    2. Create subclasses:
        ◦ DropMissing
        ◦ FillMean
        ◦ FillMode
    3. Create a DataCleaner class that applies the chosen strategy	.

In [2]:
from abc import ABC, abstractmethod

class MissingValueStrategy(ABC):
    def __init__(self, df):
        self.df = df
    @abstractmethod
    def handle(self):
        pass

class DropMissing(MissingValueStrategy):
    def handle(self):
        self.df = self.df.dropna()
        return self.df 

class FillMean(MissingValueStrategy):
    def handle(self): # self.df.mean() = show mean of all column
        return self.df.fillna(self.df.mean(numeric_only=True))  

class FillMode(MissingValueStrategy):
    def handle(self):
        return self.df.fillna(self.df.mode(numeric_only=True))

class DataCleaner:
    def __init__(self, method):
        self.method = method

    def choose(self):
        return self.method
    
DataCleaner = DataCleaner(method=FillMode(df).handle())
DataCleaner.choose()

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,29.0,172.0,60.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,88.0
3,4,David,35.0,168.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


### Exercise 3: Add Decorators for Logging and Timing

Objective:

    • Learn the Decorator Pattern to add reusable functionality (logging, timing).
    • Understand how decorators support the “Open/Closed Principle”.

Concept:
Decorators wrap functions to extend their behavior without altering the original code.

Instructions:

    1. Create two decorators:
        ◦ @log_action → logs when a method starts and finishes.
        ◦ @log_time → calculates and prints execution time.
    2. Apply them to methods in CSVReader or DataCleaner.

In [3]:
import time

def log_action(func):
    def wrapper(self, *args, **kwargs):
        print("Before method execution")
        res = func(self, *args, **kwargs)
        print("After method execution")
        return res
    return wrapper

def log_time(func):
    def wrapper(self, *args, **kwargs):
        start = time.time()
        print(f"Starting time: {start}")
        res = func(self, *args, **kwargs)
        end = time.time()
        print(f"Ending time: {end}")
        print(f"Execution Time : {end - start:.5f} seconds")
        return res
    return wrapper

class CSVReader:
    def __init__(self, file_path: str):
        self.file_path = file_path

    @log_action
    def read(self):
        df = pd.read_csv(self.file_path)
        return df

    @log_time
    def preview(self, df, n):
        return df.head(n)

rcsv = CSVReader(file_path="./sample_data.csv")
df = rcsv.read()
df = rcsv.preview(df, 5)


Before method execution
After method execution
Starting time: 1763192425.366359
Ending time: 1763192425.3664155
Execution Time : 0.00006 seconds


### Exercise 4: Implement Factory Pattern for Data Transformations
Objective:

    • Practice Factory Pattern for scalable creation of transformation objects.
    • Apply abstraction and composition.

Concept:
Instead of manually creating objects, use a factory that decides what transformation to apply.

Instructions:

    1. Create an abstract class DataTransform with a method apply(df).
    2. Implement subclasses:
        ◦ NormalizeColumns
        ◦ RemoveDuplicates
        ◦ StandardizeText
    3. Create a TransformFactory that returns transformation objects based on string input.

In [5]:
class DataTransform(ABC):
    def __init__(self, df):
        self.df = df
    
    @abstractmethod
    def apply(df):
        pass
        

class NormalizeColumns(DataTransform):
    def apply(self):
        # simple example: scale numeric columns to 0-1
        numeric_cols = self.df.select_dtypes(include="number").columns
        self.df[numeric_cols] = (self.df[numeric_cols] - self.df[numeric_cols].min()) / (
            self.df[numeric_cols].max() - self.df[numeric_cols].min()
        )
        return self.df
    
class RemoveDuplicates(DataTransform):
    def apply(self):
        return self.df.drop_duplicates()
    
class StandardizeText(DataTransform):
    def apply(self):
        # simple example: lower-case all string columns
        text_cols = self.df.select_dtypes(include="object").columns
        for col in text_cols:
            self.df[col] = self.df[col].str.lower()
        return self.df
    
class TransformFactory:
    def choose_method(self, df, method):
        if method == "normalize":
            return NormalizeColumns(df)
        
        elif method == "remove_duplicate":
            return RemoveDuplicates(df)
        
        else:
            return StandardizeText(df)
        
factory = TransformFactory()
df_cleaned = factory.choose_method(df, "normalize").apply()
df_cleaned

,id,name,age,height_cm,weight_kg,city,score
0,0.00,Alice,0.25,0.416667,0.52,New York,0.666667
1,0.25,Bob,NaN,1.000000,NaN,Los Angeles,1.000000
2,0.50,Charlie,1.00,0.666667,0.68,Chicago,NaN
3,0.75,David,NaN,NaN,1.00,Houston,0.000000
4,1.00,Eva,0.00,0.000000,0.00,New York,0.866667


### Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)
Objective:

    • Combine all previous concepts into one cleaning pipeline.
    • Use the Template Method Pattern to define a consistent workflow.

Concept:
The Template Method defines the skeleton of a process and lets subclasses override specific steps.

Instructions:

    1. Create an abstract class DataPipeline with a run() method defining steps:
        ◦ load()
        ◦ clean()
        ◦ transform()
        ◦ save()
    2. Create CSVDataPipeline that implements each step using previous classes.


In [ ]:
class DataPipeline(ABC):

    def run(self):
        """Defines the overall pipeline workflow."""
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        print("Pipeline completed successfully!")

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self, df):
        pass

    @abstractmethod
    def transform(self, df):
        pass

    @abstractmethod
    def save(self, df):
        pass

class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path

    def load(self):
        print(f"Loading data from {self.input_path}")
        return pd.read_csv(self.input_path)

    def clean(self, df):
        print("Cleaning data...")
        #drop duplicate
        df = df.drop_duplicates()
        # forward fill
        df = df.ffill()
        return df

    def transform(self, df):
        print("Transforming data...")
        # lowercasing and replace white space with _
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
        return df

    def save(self, df):
        print(f"Saving cleaned data to {self.output_path}")
        df.to_csv(self.output_path, index=False)


pipeline = CSVDataPipeline("sample_data.csv", "cleaned_data.csv")
pipeline.run()

Loading data from sample_data.csv
Cleaning data...
Transforming data...
Saving cleaned data to cleaned_data.csv
Pipeline completed successfully!
